In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.dim_cliente AS
WITH versiones AS (
  SELECT *,
         LEAD(fecha_actualizacion) OVER (PARTITION BY cliente_id ORDER BY fecha_actualizacion) AS siguiente_version
  FROM workspace.riesgo_silver.clientes
)
SELECT
  concat(cliente_id, '-', date_format(fecha_actualizacion, 'yyyyMMdd')) AS cliente_sk,
  cliente_id,
  ciudad,
  ocupacion,
  ingreso_mensual,
  score_buro,
  CASE WHEN score_buro < 580 THEN '1. Menos de 580'
       WHEN score_buro < 620 THEN '2. 580 a 619'
       WHEN score_buro < 660 THEN '3. 620 a 659'
       WHEN score_buro < 700 THEN '4. 660 a 699'
       ELSE '5. 700 o más' END                                AS banda_score,
  fecha_actualizacion                                         AS vigente_desde,
  COALESCE(date_sub(siguiente_version, 1), DATE'9999-12-31')  AS vigente_hasta,
  siguiente_version IS NULL                                   AS es_actual
FROM versiones


num_affected_rows,num_inserted_rows


DIMENCIONES


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.dim_producto AS
SELECT * FROM VALUES
  ('LI', 'Libre inversión'),
  ('CC', 'Compra de cartera'),
  ('MC', 'Microcrédito')
AS t(producto, nombre_producto)


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.dim_canal AS
SELECT DISTINCT canal, initcap(canal) AS nombre_canal
FROM workspace.riesgo_silver.solicitudes


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.dim_fecha AS
SELECT
  fecha,
  year(fecha)                   AS anio,
  month(fecha)                  AS mes,
  element_at(array('Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto',
                   'Septiembre','Octubre','Noviembre','Diciembre'), month(fecha)) AS nombre_mes,
  date_format(fecha, 'yyyy-MM') AS anio_mes,
  quarter(fecha)                AS trimestre,
  fecha = last_day(fecha)       AS es_fin_de_mes
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2026-12-31', INTERVAL 1 DAY)) AS fecha)


num_affected_rows,num_inserted_rows


FACT SOLICITUDES

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.fact_solicitudes AS
SELECT
  s.solicitud_id,
  c.cliente_sk,
  s.fecha_solicitud,
  s.producto,
  s.canal,
  s.monto_solicitado,
  s.plazo_meses,
  s.score_buro,
  s.estado,
  s.motivo_rechazo,
  s.estado = 'APROBADA'      AS aprobada,
  p.prestamo_id IS NOT NULL  AS desembolsada
FROM workspace.riesgo_silver.solicitudes s
LEFT JOIN workspace.riesgo_gold.dim_cliente c
  ON c.cliente_id = s.cliente_id
 AND s.fecha_solicitud BETWEEN c.vigente_desde AND c.vigente_hasta
LEFT JOIN workspace.riesgo_silver.prestamos p
  ON p.solicitud_id = s.solicitud_id


num_affected_rows,num_inserted_rows


FACT PRSTAMOS

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.fact_prestamos AS
SELECT
  p.prestamo_id,
  p.solicitud_id,
  c.cliente_sk,
  p.producto,
  s.canal,
  p.fecha_desembolso,
  date_format(p.fecha_desembolso, 'yyyy-MM') AS cohorte,
  p.monto_desembolsado,
  p.tasa_mensual,
  p.plazo_meses,
  p.cuota_mensual
FROM workspace.riesgo_silver.prestamos p
LEFT JOIN workspace.riesgo_silver.solicitudes s
  ON s.solicitud_id = p.solicitud_id
LEFT JOIN workspace.riesgo_gold.dim_cliente c
  ON c.cliente_id = p.cliente_id
 AND p.fecha_desembolso BETWEEN c.vigente_desde AND c.vigente_hasta


num_affected_rows,num_inserted_rows


fact_cartera_mensual

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.riesgo_gold.fact_cartera_mensual AS
WITH corte AS (
  SELECT MAX(fecha_solicitud) AS fecha_corte FROM workspace.riesgo_silver.solicitudes
),
cierres AS (
  SELECT p.*, last_day(mes) AS fecha_cierre
  FROM workspace.riesgo_gold.fact_prestamos p
  CROSS JOIN corte
  LATERAL VIEW explode(sequence(trunc(p.fecha_desembolso, 'MM'), trunc(fecha_corte, 'MM'), INTERVAL 1 MONTH)) m AS mes
  WHERE last_day(mes) <= fecha_corte
),
pagos_al_cierre AS (
  SELECT c.prestamo_id, c.fecha_cierre, COUNT(g.pago_id) AS pagos_recibidos
  FROM cierres c
  LEFT JOIN workspace.riesgo_silver.pagos g
    ON g.prestamo_id = c.prestamo_id
   AND g.fecha_pago <= c.fecha_cierre
  GROUP BY c.prestamo_id, c.fecha_cierre
),
estado AS (
  SELECT
    c.*,
    CAST(months_between(trunc(c.fecha_cierre, 'MM'), trunc(c.fecha_desembolso, 'MM')) AS INT) AS meses_de_vida,
    LEAST(a.pagos_recibidos, c.plazo_meses)                                                  AS cuotas_pagadas
  FROM cierres c
  JOIN pagos_al_cierre a
    ON a.prestamo_id = c.prestamo_id AND a.fecha_cierre = c.fecha_cierre
),
mora AS (
  SELECT
    *,
    LEAST(meses_de_vida, plazo_meses) AS cuotas_vencidas,
    CASE WHEN LEAST(meses_de_vida, plazo_meses) > cuotas_pagadas
         THEN datediff(fecha_cierre, add_months(fecha_desembolso, cuotas_pagadas + 1))
         ELSE 0 END                   AS dias_mora
  FROM estado
  WHERE cuotas_pagadas < plazo_meses
)
SELECT
  prestamo_id,
  fecha_cierre,
  cliente_sk,
  producto,
  canal,
  cohorte,
  meses_de_vida,
  cuotas_vencidas,
  cuotas_pagadas,
  GREATEST(cuotas_vencidas - cuotas_pagadas, 0) AS cuotas_en_atraso,
  dias_mora,
  CASE WHEN dias_mora = 0   THEN 'Al día'
       WHEN dias_mora <= 30 THEN '1-30'
       WHEN dias_mora <= 60 THEN '31-60'
       WHEN dias_mora <= 90 THEN '61-90'
       ELSE 'Más de 90' END                     AS bucket_mora,
  CASE WHEN dias_mora = 0   THEN 1
       WHEN dias_mora <= 30 THEN 2
       WHEN dias_mora <= 60 THEN 3
       WHEN dias_mora <= 90 THEN 4
       ELSE 5 END                               AS bucket_orden,
  CAST(monto_desembolsado
       * (pow(1 + tasa_mensual, plazo_meses) - pow(1 + tasa_mensual, cuotas_pagadas))
       / (pow(1 + tasa_mensual, plazo_meses) - 1) AS DECIMAL(18,2)) AS saldo_capital
FROM mora


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT bucket_mora,
       ROUND(100 * SUM(saldo_capital) / SUM(SUM(saldo_capital)) OVER (), 1) AS pct_saldo
FROM workspace.riesgo_gold.fact_cartera_mensual
WHERE fecha_cierre = '2025-03-31'
GROUP BY bucket_mora, bucket_orden
ORDER BY bucket_orden


bucket_mora,pct_saldo
Al día,96.6
1-30,3.2
31-60,0.2


In [0]:
%sql

Select count(*) from workspace.riesgo_gold.fact_cartera_mensual

count(*)
3051
